# PUMA-themed sales practice
Simulated data. Run cells in order. Read guides/01_Start_here_and_data_dictionary.md first.


## 01. Imports and folders

Path represents a filesystem path without manually joining slash characters. json reads the answer checks; sqlite3 reads the SQL solution; pandas handles tables; NumPy provides NaN and numerical checks; Matplotlib draws figures. ROOT is the top-level project folder, DATA the raw-data folder and OUT the output folder. mkdir creates the output folder if needed. The script uses __file__ to locate itself; the notebook uses its current folder.

Checkpoint: After running, print ROOT and DATA. Confirm DATA points to the folder containing sales_raw.csv.

Practice: Change OUT to a different practice output subfolder and rerun; do not point it at the data folder.


In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Save charts without requiring an interactive display.
import matplotlib.pyplot as plt

ROOT=Path.cwd()
if ROOT.name=='python':
    ROOT=ROOT.parent
if not (ROOT/'data'/'sales_raw.csv').exists():
    raise FileNotFoundError('Open this notebook from the extracted project folder or its python subfolder.')
DATA=ROOT/'data'
OUT=ROOT/'results'/'python'
OUT.mkdir(parents=True,exist_ok=True)



## 02. Load and inspect the sales table

read_csv parses a CSV into a DataFrame. head shows the first five records. shape returns row and column counts. dtypes shows storage types. isna().sum() counts missing values per column. to_datetime converts the date column using an explicit YYYY-MM-DD format. errors=raise stops on invalid dates in the clean file. is_unique checks line identifiers. nunique counts unique order IDs.

Checkpoint: Expect shape (7200, 19), no missing raw values, 7,200 unique LineID values and 3,600 unique OrderID values.

Practice: Print the last five rows with tail and inspect Quantity with describe.


In [ ]:
sales=pd.read_csv(DATA/'sales_raw.csv')
print(sales.head())
print('Rows and columns:',sales.shape)
print(sales.dtypes)
print('Missing values:',sales.isna().sum().to_dict())
sales['OrderDate']=pd.to_datetime(sales['OrderDate'],format='%Y-%m-%d',errors='raise')
assert len(sales)==7200
assert sales['LineID'].is_unique
assert sales['OrderID'].nunique()==3600



## 03. Validate business rules

gt, ge, le and between create boolean tests. all requires every row to satisfy the condition. The return quantity must be within the originally sold quantity. groupby collects rows by order, then sum adds their flags. eq(1).all() verifies exactly one flagged row per order. These tests distinguish a valid multi-line order from a duplicate sales record.

Checkpoint: The cell should finish without an AssertionError. A failure needs investigation before profit calculation.

Practice: Add validations for nonnegative ShippingCostEUR, UnitCostEUR and DeliveryDays.


In [ ]:
assert sales['Quantity'].gt(0).all()
assert sales['ReturnedQty'].ge(0).all()
assert sales['ReturnedQty'].le(sales['Quantity']).all()
assert sales['DiscountRate'].between(0,1).all()
assert sales['UnitPriceEUR'].gt(0).all()
assert sales.groupby('OrderID')['FirstOrderFlag'].sum().eq(1).all()



## 04. Clean the dirty practice file

drop_duplicates removes only the exact duplicated record in this exercise. str.strip removes surrounding spaces. str.lower standardises comparison case, and map applies the approved channel names. errors=coerce turns invalid dates or numerical text into missing values so they can be flagged. set contains allowed keys. isin tests membership. A DataFrame of quality rules records several possible issues per row. any(axis=1) finds rows with at least one issue. loc separates rejects from accepted rows, and join attaches diagnostic flags using aligned indexes.

Checkpoint: From 121 input rows, remove one exact duplicate, then retain 114 accepted rows and log 6 rejected rows. Check dirty_rejected_rows.csv. Blank discount is rejected, not guessed.

Practice: Add a required CustomerID check and explain how you would handle conflicting rows with the same LineID.


In [ ]:
dirty=pd.read_csv(DATA/'sales_dirty_practice.csv')
duplicate_count=int(dirty.duplicated(subset=['LineID']).sum())
# This file has one exact duplicate. Conflicting repeated IDs require investigation.
dirty=dirty.drop_duplicates().copy()
dirty['Country']=dirty['Country'].str.strip()
channel_map={'wholesale':'Wholesale','retail':'Retail','e-commerce':'E-commerce'}
dirty['Channel']=dirty['Channel'].str.strip().str.lower().map(channel_map)
dirty['OrderDate']=pd.to_datetime(dirty['OrderDate'],format='%Y-%m-%d',errors='coerce')
for c in ['Quantity','ReturnedQty','UnitPriceEUR','DiscountRate']:
    dirty[c]=pd.to_numeric(dirty[c],errors='coerce')
valid_products=set(pd.read_csv(DATA/'products.csv')['ProductID'])
valid_countries=set(pd.read_csv(DATA/'countries.csv')['Country'])
checks=pd.DataFrame({
    'missing_required':dirty[['OrderDate','Quantity','ReturnedQty','UnitPriceEUR','DiscountRate','Channel']].isna().any(axis=1),
    'invalid_quantity':dirty['Quantity'].le(0),
    'invalid_returns':dirty['ReturnedQty'].lt(0)|dirty['ReturnedQty'].gt(dirty['Quantity']),
    'invalid_price':dirty['UnitPriceEUR'].le(0),
    'invalid_discount':~dirty['DiscountRate'].between(0,1),
    'unknown_product':~dirty['ProductID'].isin(valid_products),
    'unknown_country':~dirty['Country'].isin(valid_countries)
})
bad=checks.any(axis=1)
rejects=dirty.loc[bad].join(checks.loc[bad].add_prefix('Issue_'))
clean_practice=dirty.loc[~bad].copy()
rejects.to_csv(OUT/'dirty_rejected_rows.csv',index=False)
clean_practice.to_csv(OUT/'dirty_accepted_rows.csv',index=False)
print('Dirty-file duplicate count:',duplicate_count,'Accepted:',len(clean_practice),'Rejected:',len(rejects))
assert duplicate_count==1 and len(clean_practice)==114 and len(rejects)==6
# The small cleaning file is a separate exercise. Do not append it to sales.



## 05. Calculate time keys and revenue

dt.year extracts a year from datetime values. to_period(M) groups timestamps into calendar months, then to_timestamp converts those month keys to first-day timestamps. Column arithmetic is vectorised: pandas applies it across all rows without a Python loop. round(2) keeps money components at cents. Refunds use the discounted unit price. Calculate net sales only after the earlier components exist.

Checkpoint: Compare one returned line with its hand calculation and with Excel columns V:Z. Annual 2024 net sales will be 1,885,059.75.

Practice: Print five rows where ReturnedQty is greater than zero and inspect refund values.


In [ ]:
sales['Year']=sales['OrderDate'].dt.year
sales['MonthStart']=sales['OrderDate'].dt.to_period('M').dt.to_timestamp()
sales['GrossSalesEUR']=(sales['Quantity']*sales['UnitPriceEUR']).round(2)
sales['DiscountEUR']=(sales['GrossSalesEUR']*sales['DiscountRate']).round(2)
sales['SalesAfterDiscountEUR']=sales['GrossSalesEUR']-sales['DiscountEUR']
sales['RefundEUR']=(sales['ReturnedQty']*sales['UnitPriceEUR']*(1-sales['DiscountRate'])).round(2)
sales['NetSalesEUR']=(sales['SalesAfterDiscountEUR']-sales['RefundEUR']).round(2)



## 06. Calculate profit and flags

Net units remove returned units. COGS uses those retained units and the historical transaction cost. Gross profit subtracts COGS; contribution subtracts the allocated shipping cost. replace(0,np.nan) makes zero-denominator margins unavailable instead of causing invalid division. gt(0).astype(int) converts return presence into a 1/0 flag. This line margin is useful for inspection but must not be averaged to calculate overall margin.

Checkpoint: Verify gross profit equals net sales minus COGS for sample lines. Verify a returned unit reverses cost under the documented assumption.

Practice: Create a flag for lines with gross margin below 40%, treating missing margins separately.


In [ ]:
sales['NetUnits']=sales['Quantity']-sales['ReturnedQty']
sales['COGSEUR']=(sales['NetUnits']*sales['UnitCostEUR']).round(2)
sales['GrossProfitEUR']=(sales['NetSalesEUR']-sales['COGSEUR']).round(2)
sales['ContributionEUR']=(sales['GrossProfitEUR']-sales['ShippingCostEUR']).round(2)
sales['LineMargin']=sales['GrossProfitEUR'].div(sales['NetSalesEUR'].replace(0,np.nan))
sales['ReturnFlag']=sales['ReturnedQty'].gt(0).astype(int)



## 07. Filter rows with conditions

loc takes a boolean mask. eq compares a column to a value. Multiple masks joined by & require every condition. copy creates an independent working object for sales_2024 so later edits do not rely on view/copy behaviour. The Poland online subset requires the same year, country and channel together.

Checkpoint: Poland E-commerce net sales for 2024 equals 17,898.50 EUR in this dataset.

Practice: Find India Retail net sales for 2023 using the same pattern.


In [ ]:
sales_2024=sales.loc[sales['Year'].eq(2024)].copy()
poland_online=sales.loc[sales['Year'].eq(2024)&sales['Country'].eq('Poland')&sales['Channel'].eq('E-commerce')]
print('Poland online 2024 sales:',round(poland_online['NetSalesEUR'].sum(),2))



## 08. Aggregate annual KPIs

groupby(Year,as_index=False) produces one row per year and keeps Year as an ordinary column. agg uses named output columns, each mapped to a source column and operation. sum is correct for additive money and units; nunique is correct for order count. Rates are calculated after aggregation from matching totals. sort_values guarantees chronology before pct_change compares each year with its predecessor. The first year has no prior observation and remains NaN.

Checkpoint: 2024 has 1,800 placed orders, gross profit 920,398.35, gross margin about 0.48825951 and unit return rate about 0.01690906.

Practice: Add total RefundEUR and calculate sales after discount divided by orders as a separately named measure.


In [ ]:
annual=sales.groupby('Year',as_index=False).agg(
    NetSalesEUR=('NetSalesEUR','sum'),GrossProfitEUR=('GrossProfitEUR','sum'),
    ContributionEUR=('ContributionEUR','sum'),Orders=('OrderID','nunique'),
    Quantity=('Quantity','sum'),ReturnedQty=('ReturnedQty','sum'),NetUnits=('NetUnits','sum'))
annual['GrossMargin']=annual['GrossProfitEUR']/annual['NetSalesEUR']
annual['ReturnRate']=annual['ReturnedQty']/annual['Quantity']
annual['AverageOrderValueEUR']=annual['NetSalesEUR']/annual['Orders']
annual=annual.sort_values('Year')
annual['YoYGrowth']=annual['NetSalesEUR'].pct_change(fill_method=None)
print(annual.to_string(index=False))



## 09. Build monthly trends

Group by both year and the real month key. cumsum accumulates sales within each year, restarting in January. rolling(3,min_periods=1).mean() computes an average using up to three observations, so early months use shorter windows. The moving average crosses year boundaries; YTD does not. pct_change compares previous observed rows. This dataset has no missing months, so previous observed month equals previous calendar month.

Checkpoint: There are 24 monthly rows. December YTDSales must match its annual total. January 2023 MoMGrowth is missing.

Practice: Create a separate year-partitioned moving average and explain the difference at January 2024.


In [ ]:
monthly=sales.groupby(['Year','MonthStart'],as_index=False).agg(
    NetSalesEUR=('NetSalesEUR','sum'),GrossProfitEUR=('GrossProfitEUR','sum'))
monthly=monthly.sort_values('MonthStart')
monthly['MoMGrowth']=monthly['NetSalesEUR'].pct_change(fill_method=None)
monthly['YTDSales']=monthly.groupby('Year')['NetSalesEUR'].cumsum()
monthly['ThreeMonthAverage']=monthly['NetSalesEUR'].rolling(3,min_periods=1).mean()



## 10. Rank countries and calculate category share

The country table groups only 2024 rows and sorts descending by sales with country as the tie-breaker. Category share divides each category by the same 2024 all-region total. GrossMargin uses category-level profit and category-level net sales. Sorting changes display order without changing the underlying measures.

Checkpoint: Country has 12 rows; category has 3 rows; category shares add to 1 before display rounding.

Practice: Return the top three countries and their combined share of 2024 sales.


In [ ]:
country=sales_2024.groupby('Country',as_index=False).agg(NetSalesEUR=('NetSalesEUR','sum'),Orders=('OrderID','nunique'))
country=country.sort_values(['NetSalesEUR','Country'],ascending=[False,True])
category=sales_2024.groupby('Category',as_index=False).agg(NetSalesEUR=('NetSalesEUR','sum'),GrossProfitEUR=('GrossProfitEUR','sum'))
category['Share']=category['NetSalesEUR']/category['NetSalesEUR'].sum()
category['GrossMargin']=category['GrossProfitEUR']/category['NetSalesEUR']



## 11. Merge a dimension with validation

merge joins customer descriptions to sales using CustomerID. how=left preserves sales rows. validate=many_to_one requires the customer lookup key to be unique, preventing row multiplication. indicator adds _merge so you can inspect matched versus unmatched keys. The assertions verify every line survived and found a customer. Segment counts use active purchasers, not every record in the dimension.

Checkpoint: The enriched table still has 7,200 lines and every _merge value is both.

Practice: Use the same checks to merge product descriptions to raw order_lines.csv.


In [ ]:
customers=pd.read_csv(DATA/'customers.csv')
enriched=sales.merge(customers,on='CustomerID',how='left',validate='many_to_one',indicator=True)
assert len(enriched)==len(sales)
assert enriched['_merge'].eq('both').all()
segments=enriched.loc[enriched['Year'].eq(2024)].groupby('CustomerSegment',as_index=False).agg(
    NetSalesEUR=('NetSalesEUR','sum'),ActiveCustomers=('CustomerID','nunique'),Orders=('OrderID','nunique'))



## 12. Pivot regions and years

pivot_table reshapes long data into a wide regional comparison. index supplies row labels, columns supplies years and values supplies the sales measure. aggfunc=sum is required because there are many lines for every region/year combination. Dividing the 2024 column by 2023 and subtracting 1 calculates regional growth. A pivot changes shape, not the meaning of the amount.

Checkpoint: Expect three regional rows. EMEA sales are 594,604.75 in 2023 and 668,517.25 in 2024.

Practice: Add ChangeEUR = sales in 2024 minus sales in 2023.


In [ ]:
region_year=sales.pivot_table(index='Region',columns='Year',values='NetSalesEUR',aggfunc='sum')
region_year['YoYGrowth']=region_year[2024]/region_year[2023]-1
print(region_year)



## 13. Compare actual sales with targets

Read target MonthStart as a date so its type matches the grouped actual table. Aggregate sales to MonthStart and Region before the merge. validate=one_to_one requires one record on each side for the composite key. The _merge assertion checks coverage. VarianceEUR is a signed money difference; Attainment is a ratio. Filtering below 1 finds shortfalls.

Checkpoint: The merged table has 72 rows. Total 2024 target equals 1,967,911.08 EUR.

Practice: Aggregate annual actual and target by region, then calculate attainment from those annual totals.


In [ ]:
targets=pd.read_csv(DATA/'monthly_targets.csv',parse_dates=['MonthStart'])
actual=sales.groupby(['MonthStart','Region'],as_index=False).agg(NetSalesEUR=('NetSalesEUR','sum'))
target_comparison=targets.merge(actual,on=['MonthStart','Region'],how='left',validate='one_to_one',indicator=True)
assert target_comparison['_merge'].eq('both').all()
target_comparison['VarianceEUR']=target_comparison['NetSalesEUR']-target_comparison['TargetNetSalesEUR']
target_comparison['Attainment']=target_comparison['NetSalesEUR']/target_comparison['TargetNetSalesEUR']
under_target=target_comparison.loc[target_comparison['Attainment'].lt(1)].sort_values('VarianceEUR')



## 14. Analyse returns and delivery

The returns table tracks sold units, returned units, refund money, returned lines and total lines separately. UnitReturnRate and ReturnedLineRate have different denominators and answer different questions. assign adds a LateLine boolean, and mean of booleans gives the proportion of True values. DeliveryDays is a line-level measure, so the report names it accordingly. The seven-day rule is a practice threshold.

Checkpoint: Overall 2024 returned units are 579 from 34,242 sold units. Delivery output has one row per channel.

Practice: Build an order-level late flag with groupby(OrderID).max, then compare its rate to the line-level result.


In [ ]:
returns=sales_2024.groupby('Category',as_index=False).agg(
    SoldUnits=('Quantity','sum'),ReturnedUnits=('ReturnedQty','sum'),RefundEUR=('RefundEUR','sum'),ReturnedLines=('ReturnFlag','sum'),Lines=('LineID','count'))
returns['UnitReturnRate']=returns['ReturnedUnits']/returns['SoldUnits']
returns['ReturnedLineRate']=returns['ReturnedLines']/returns['Lines']
delivery=sales_2024.assign(LateLine=sales_2024['DeliveryDays'].gt(7)).groupby('Channel',as_index=False).agg(
    AvgLineDeliveryDays=('DeliveryDays','mean'),LateLineRate=('LateLine','mean'))



## 15. Build customer RFM

First aggregate the lines into unique customer/order records. Then find the most recent order, count orders and add monetary value per customer. pd.Timestamp creates a fixed reference date. Subtracting dates creates timedeltas; dt.days extracts whole day differences. Frequency means placed orders during 2024; customers without 2024 orders are absent. No unsupported scoring rules are added.

Checkpoint: No RecencyDays value should be negative. Sum MonetaryEUR to reconcile to total 2024 sales.

Practice: Find customers with frequency at least 2 and recency over 90 days.


In [ ]:
order_totals=sales_2024.groupby(['CustomerID','OrderID'],as_index=False).agg(
    OrderDate=('OrderDate','max'),NetSalesEUR=('NetSalesEUR','sum'))
rfm=order_totals.groupby('CustomerID',as_index=False).agg(
    LastOrderDate=('OrderDate','max'),Frequency=('OrderID','nunique'),MonetaryEUR=('NetSalesEUR','sum'))
rfm['RecencyDays']=(pd.Timestamp('2024-12-31')-rfm['LastOrderDate']).dt.days
rfm=rfm.sort_values('MonetaryEUR',ascending=False)



## 16. Flag large values for review

quantile(.25) and quantile(.75) produce the lower and upper quartiles. The interquartile range is q3-q1. The upper review threshold is q3 + 1.5 times that range. The mask identifies large order totals. This statistical rule flags unusual values; it does not prove fraud, incorrect data or a reason to remove the orders. Channel context matters because Wholesale quantities are larger by design.

Checkpoint: Inspect the flagged orders and relate their size to their channel before interpreting them.

Practice: Calculate separate thresholds per channel and compare how many orders are flagged.


In [ ]:
q1=order_totals['NetSalesEUR'].quantile(.25)
q3=order_totals['NetSalesEUR'].quantile(.75)
upper=q3+1.5*(q3-q1)
large_orders=order_totals.loc[order_totals['NetSalesEUR'].gt(upper)].copy()
# These are review candidates. Wholesale orders can be legitimately large.



## 17. Write a reusable function and scenario

def declares a function. Its parameters frame and group_column allow the same grouping logic to be reused for Channel, Country or Category. return supplies the completed summary. In the scenario, copy protects the actual values. Adding .05 increases discounts by five percentage points. clip caps the rate at 100%. Fixed net units, unit prices and costs isolate the arithmetic effect. This scenario is not a forecast of customer demand.

Checkpoint: Scenario sales should be lower than current sales because the discount increases and volume is fixed.

Practice: Call summarise(sales_2024,'Country') and calculate scenario gross profit.


In [ ]:
def summarise(frame,group_column):
    result=frame.groupby(group_column,as_index=False).agg(
        NetSalesEUR=('NetSalesEUR','sum'),GrossProfitEUR=('GrossProfitEUR','sum'))
    result['GrossMargin']=result['GrossProfitEUR'].div(result['NetSalesEUR'].replace(0,np.nan))
    return result.sort_values('NetSalesEUR',ascending=False)

channel_summary=summarise(sales_2024,'Channel')
scenario=sales_2024.copy()
scenario['ScenarioDiscount']=(scenario['DiscountRate']+.05).clip(upper=1)
scenario['ScenarioNetSalesEUR']=(scenario['NetUnits']*scenario['UnitPriceEUR']*(1-scenario['ScenarioDiscount'])).round(2)
scenario_impact=scenario.groupby('Category',as_index=False).agg(
    CurrentNetSalesEUR=('NetSalesEUR','sum'),ScenarioNetSalesEUR=('ScenarioNetSalesEUR','sum'))
scenario_impact['ChangeEUR']=scenario_impact['ScenarioNetSalesEUR']-scenario_impact['CurrentNetSalesEUR']



## 18. Reconcile Python with controls and SQL

json.loads reads independently calculated check totals. itertuples loops through annual summary rows with named fields. Each assertion compares selected values and order counts. The with statement closes the SQLite connection after use. read_sql_query loads annual SQL totals into a DataFrame. merge aligns years before comparison. np.allclose checks money within 0.01 using rtol=0, so relative tolerance cannot conceal a material mismatch.

Checkpoint: All assertions should pass. A difference requires checking population, dates, joins, refund logic and rounding.

Practice: Add a SQL gross-profit comparison and a contribution bridge.


In [ ]:
checks=json.loads((ROOT/'results'/'expected_checks.json').read_text())
for row in annual.itertuples(index=False):
    expected=checks['years'][str(row.Year)]
    assert abs(row.NetSalesEUR-expected['NetSalesEUR'])<.01
    assert abs(row.GrossProfitEUR-expected['GrossProfitEUR'])<.01
    assert row.Orders==expected['FirstOrderFlag']
with sqlite3.connect(ROOT/'sql'/'puma_solution.sqlite') as con:
    sql_annual=pd.read_sql_query('SELECT Year,SUM(NetSalesEUR) AS SQLSales FROM v_sales GROUP BY Year',con)
comparison=annual.merge(sql_annual,on='Year',validate='one_to_one')
assert np.allclose(comparison['NetSalesEUR'],comparison['SQLSales'],atol=.01,rtol=0)
assert abs(sales['GrossSalesEUR'].sum()-sales['DiscountEUR'].sum()-sales['RefundEUR'].sum()-sales['NetSalesEUR'].sum())<.01



## 19. Draw the dashboard

subplots creates a figure and a 2×2 array of axes. Iterating grouped monthly rows draws one line for each year. Values are divided by 1000 only for chart display and labelled EUR thousands. head(6) deliberately selects a top-six country chart; the country table still preserves all countries. The final panel compares monthly actual with the artificial target. savefig writes a PNG, and close releases the figure. Agg allows the script to save charts without a GUI; Jupyter can display the saved image separately.

Checkpoint: Open results/python/dashboard.png and verify labels, years, category names and units.

Practice: Change the first panel to a grouped column chart, keeping chronology and units correct.


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,9),layout='constrained')
for year,frame in monthly.groupby('Year'):
    axes[0,0].plot(frame['MonthStart'].dt.month,frame['NetSalesEUR']/1000,marker='o',label=str(year))
axes[0,0].set(title='Monthly net sales',xlabel='Month',ylabel='EUR thousands',xticks=range(1,13));axes[0,0].legend()
top=country.head(6).sort_values('NetSalesEUR')
axes[0,1].barh(top['Country'],top['NetSalesEUR']/1000,color='#B3282D')
axes[0,1].set(title='Top six countries, 2024',xlabel='EUR thousands')
cat=category.sort_values('NetSalesEUR',ascending=False)
axes[1,0].bar(cat['Category'],cat['NetSalesEUR']/1000,color='#253746')
axes[1,0].set(title='Category net sales, 2024',ylabel='EUR thousands')
tc=target_comparison.loc[target_comparison['MonthStart'].dt.year.eq(2024)].groupby('MonthStart',as_index=False)[['NetSalesEUR','TargetNetSalesEUR']].sum()
axes[1,1].plot(tc['MonthStart'].dt.month,tc['NetSalesEUR']/1000,label='Actual')
axes[1,1].plot(tc['MonthStart'].dt.month,tc['TargetNetSalesEUR']/1000,label='Target',linestyle='--')
axes[1,1].set(title='Actual versus artificial target, 2024',xlabel='Month',ylabel='EUR thousands',xticks=range(1,13));axes[1,1].legend()
for ax in axes.flat:
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',alpha=.2)
fig.suptitle('PUMA-themed sales practice | Simulated data',fontsize=16)
fig.savefig(OUT/'dashboard.png',dpi=160)
plt.close(fig)



## 20. Export the analysis tables

A dictionary maps output filenames to completed DataFrames. The for loop visits each pair and to_csv saves an analysis result. index=False avoids adding the DataFrame row index as an extra business field. The full calculated sales table is exported separately. Raw files remain untouched. Save your script or notebook with the same folder structure so rerunning regenerates the results.

Checkpoint: Expect the named analysis CSVs plus calculated sales and cleaning logs in results/python.

Practice: Write a five-sentence business review using actual output numbers and explicit simulated-data scope.


In [ ]:
outputs={'annual':annual,'monthly':monthly,'country_2024':country,'category_2024':category,'customer_segments':segments,
    'target_comparison':target_comparison,'under_target':under_target,'returns_2024':returns,'delivery_2024':delivery,
    'customer_rfm_2024':rfm,'large_orders_review':large_orders,'channel_2024':channel_summary,'discount_scenario':scenario_impact}
for name,frame in outputs.items():
    frame.to_csv(OUT/f'{name}.csv',index=False)
sales.to_csv(OUT/'sales_calculated.csv',index=False)
print('Complete: all annual controls and SQL comparisons passed. Reports are in results/python.')
